# 最小评测 Pipeline · 自建 mini benchmark

**目标**：演示 *业务自建评测* 的最小骨架：

1. 任务集（10 道多跳 QA，沿用第 04 章迷你 KB）。
2. 三个 agent baseline：Naive RAG / ReAct / ReWOO（来自前几章实现，简化版）。
3. 评分：先用 *精确匹配*；再用 *LLM-as-Judge* 评开放回答。
4. 输出 leaderboard + 错误分析。

把这个 notebook 当作业务评测脚手架的模板。

In [ ]:
import os, sys, json, re, statistics, numpy as np
sys.path.append(os.path.abspath('../..'))
from anthropic import Anthropic
from utils.llm_client import LLMClient
client = LLMClient(temperature=0)
anthropic = Anthropic()
MODEL = client.model

In [ ]:
DOCS = [
    'ReAct（Yao 2022）让 LLM 交替输出 Thought/Action/Observation。',
    'Reflexion（Shinn 2023）通过自然语言反思在多 episode 间积累经验。',
    'LATS（Zhou 2024）= ToT + ReAct + Reflexion，用 MCTS 串起搜索/行动/反思。',
    'MCP（Anthropic 2024）把 LLM 与工具/数据接口标准化。',
    'GRPO（DeepSeek 2024）省掉 value model，用组内归一 advantage。',
    'DAPO（ByteDance 2025）GRPO 改进版。',
    'SkyRL-Agent（2025）多轮长程 agent RL 训练框架。',
    'MapAgent（2025）分层多 agent 框架，map-tool agent 并行调地图 API。',
    'Generative Agents（Park 2023）三层记忆：观察、反思、计划。',
    'Self-RAG（Asai 2024）让 LLM 输出 reflection token 决定检索/引用。',
]

def search(query: str, k: int = 3) -> list[str]:
    scored = sorted(DOCS, key=lambda d: -sum(1 for w in query if w in d))
    return scored[:k]

EVAL = [
    {'id': 'q1', 'q': 'LATS 是哪三种方法的结合？', 'expect': ['ToT', 'ReAct', 'Reflexion']},
    {'id': 'q2', 'q': 'GRPO 相比 PPO 的关键区别？', 'expect': ['不需要', 'value', '组内']},
    {'id': 'q3', 'q': 'MCP 是谁提出的？', 'expect': ['Anthropic']},
    {'id': 'q4', 'q': 'Generative Agents 的三层记忆？', 'expect': ['观察', '反思', '计划']},
    {'id': 'q5', 'q': 'SkyRL-Agent 适合什么任务？', 'expect': ['多轮', '长程']},
    {'id': 'q6', 'q': 'Self-RAG 用什么 token 决定是否检索？', 'expect': ['reflection']},
    {'id': 'q7', 'q': 'MapAgent 是分层架构还是单层？', 'expect': ['分层']},
    {'id': 'q8', 'q': 'DAPO 是哪个公司提出的？', 'expect': ['ByteDance', '字节']},
    {'id': 'q9', 'q': 'Reflexion 的反思以什么形式存在？', 'expect': ['自然语言']},
    {'id': 'q10', 'q': 'ReAct 中的三种 token 类型是什么？', 'expect': ['Thought', 'Action', 'Observation']},
]

## 1. 三个 baseline

In [ ]:
def naive_rag(q: str) -> str:
    ctx = '\n'.join(search(q, 3))
    prompt = f'仅根据下面证据简短回答：\n{ctx}\n\n问题：{q}'
    return client.chat([{'role': 'user', 'content': prompt}])['text']

TOOLS = [{
    'name': 'search', 'description': '检索 LLM Agent 知识库',
    'input_schema': {'type': 'object', 'properties': {'query': {'type': 'string'}, 'k': {'type': 'integer'}}, 'required': ['query']},
}]

def react_agent(q: str, max_steps: int = 5) -> str:
    msgs = [{'role': 'user', 'content': q}]
    for _ in range(max_steps):
        r = anthropic.messages.create(model=MODEL, max_tokens=400, tools=TOOLS, messages=msgs)
        if r.stop_reason != 'tool_use':
            return ''.join(b.text for b in r.content if b.type == 'text')
        msgs.append({'role': 'assistant', 'content': r.content})
        results = []
        for b in r.content:
            if b.type == 'tool_use':
                hits = search(**b.input)
                results.append({'type': 'tool_result', 'tool_use_id': b.id, 'content': '\n'.join(hits)})
        msgs.append({'role': 'user', 'content': results})
    return '[max]'

BASELINES = {'naive_rag': naive_rag, 'react': react_agent}

## 2. 评分：精确匹配 + LLM-judge

In [ ]:
def em_score(pred: str, expect: list[str]) -> int:
    return int(all(e.lower() in pred.lower() for e in expect))

JUDGE_PROMPT = (
    '你是评分员。请判断模型回答是否正确覆盖参考要点。回答只能是数字 0 或 1，不要解释。\n\n'
    '问题：{q}\n'
    '参考要点：{exp}\n'
    '模型回答：{pred}\n'
    '评分：'
)

def llm_judge(q: str, pred: str, expect: list[str]) -> int:
    out = client.chat([{'role': 'user', 'content': JUDGE_PROMPT.format(
        q=q, exp=' / '.join(expect), pred=pred,
    )}])
    txt = out.strip()
    return 1 if txt.startswith('1') else 0

## 3. 跑评测 + leaderboard

In [ ]:
import time

results = {}
for name, fn in BASELINES.items():
    rows = []
    t0 = time.time()
    for item in EVAL:
        pred = fn(item['q'])
        rows.append({
            'id': item['id'], 'q': item['q'], 'pred': pred,
            'em': em_score(pred, item['expect']),
            'judge': llm_judge(item['q'], pred, item['expect']),
        })
    results[name] = {
        'rows': rows,
        'em_acc': statistics.mean(r['em'] for r in rows),
        'judge_acc': statistics.mean(r['judge'] for r in rows),
        'wall': time.time() - t0,
    }

for name, r in results.items():
    print(f"{name:12s}  EM={r['em_acc']:.2f}  Judge={r['judge_acc']:.2f}  wall={r['wall']:.1f}s")

## 4. 错误分析

把 EM=0 但 Judge=1 的 case 列出来——这往往是「答案对但措辞不同」，提示 EM 太苛刻。

In [ ]:
for name, r in results.items():
    print('===', name, '===')
    for row in r['rows']:
        if row['em'] != row['judge']:
            print(f"  [{row['id']}] EM={row['em']} J={row['judge']}  pred={row['pred'][:60]}...")

## 5. 落地建议

- *EM* 严格但偏严：业务里更适合 *LLM-judge*（成对比较 / 评分）。
- 评测必须 *版本化*：每次 prompt/模型变更就跑一遍并入库。
- 不要只看 acc：加 token、wall time、失败模式（长尾）。
- *Adversarial cases*：构造一些已知会让 agent 出错的 case，确保不退化。

## 进阶练习

1. 把 baseline 集合扩展到上文 ReWOO / Agentic RAG。
2. 用 *pairwise judge*（A vs B）替代 0/1，画 win-rate 矩阵。
3. 把 `EVAL` 持久化到 jsonl + git，每次跑评测自动追加 commit hash。